# 08 Static and heterogeneous graph models (Families D, E)
Encoders: GCN, GraphSAGE, GAT (static, payment edges) and R-GCN, HGT, HAN (heterogeneous: account/device/ip). Node2Vec is omitted: a static unsupervised embedding cannot embed accounts that appear after training, which is the common case in this task; GraphSAGE/GCN are the stronger static baselines.

Fair comparison: every encoder feeds the same payment head together with the same tabular features. The row `none` is the identical head without any graph, so graph contribution is measured like for like.

In [1]:
import sys, pathlib, json, warnings; warnings.filterwarnings('ignore')
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False})
def table(name):
    p = ROOT / 'reports' / 'tables' / name
    return pd.read_csv(p) if p.exists() else None

In [2]:
import json, time
from presage.data.prepare import prepare
from presage.models.graph_train import fit_edge_model
D = prepare(); A = D['A']; df = D['df']
t0 = time.time()
mdl, pv, pt, info = fit_edge_model('gcn', D['snaps'], D['snap_of'], A['src'], A['dst'], A['tab'], A['y_app'], D['tr'], D['va'], D['te'], epochs=5, verbose=True)
from sklearn.metrics import average_precision_score
print('GCN demo (5 epochs) test PR-AUC %.3f in %.0fs' % (average_precision_score(A['y_app'][D['te']], pt[D['te']]), time.time()-t0))

19:15:50 INFO snapshots: built 31 snapshots (15262 accounts, 17841 devices, 8000 ips)


  [gcn] ep1 loss 9.724 val PR-AUC 0.588 (19s)


  [gcn] ep2 loss 3.378 val PR-AUC 0.782 (15s)


  [gcn] ep3 loss 1.211 val PR-AUC 0.827 (21s)


  [gcn] ep4 loss 0.869 val PR-AUC 0.865 (22s)


  [gcn] ep5 loss 0.724 val PR-AUC 0.877 (28s)


GCN demo (5 epochs) test PR-AUC 0.914 in 114s


In [3]:
g = table('graph_models.csv'); g[['model','pr_auc','pr_auc_lo','pr_auc_hi','recall','precision','f2','mcc','train_s','latency_ms','n_params']].round(3)

,model,pr_auc,pr_auc_lo,pr_auc_hi,recall,precision,f2,mcc,train_s,latency_ms,n_params
0,none,0.874,0.821,0.905,0.813,0.743,0.798,0.776,4.476,0.001,24129
1,gcn,0.921,0.883,0.948,0.826,0.905,0.840,0.864,41.786,0.011,45505
2,sage,0.924,0.875,0.950,0.829,0.915,0.845,0.870,21.963,0.005,53697
3,gat,0.918,0.872,0.943,0.829,0.915,0.845,0.870,118.642,0.034,54081
4,rgcn,0.928,0.876,0.954,0.839,0.903,0.851,0.870,98.885,0.014,78449
5,hgt,0.959,0.921,0.976,0.900,0.936,0.907,0.918,383.606,0.062,161783
6,han,0.914,0.863,0.946,0.806,0.916,0.826,0.859,126.957,0.017,72257
7,tgat,0.942,0.904,0.963,0.910,0.925,0.913,0.917,101.790,0.031,72017
8,evolvegcn,0.872,0.828,0.902,0.765,0.856,0.781,0.808,46.334,0.013,95297


Reading: graph encoders add information over the tabular-only twin. Compare against the XGBoost-on-engineered-graph-features rows in notebook 06: engineered graph features plus gradient boosting is a very strong baseline, and learned graph encoders do not automatically beat it.